# Groundwater WQI Regression

This notebook uses a **simple leakage-safe regression workflow**.

### Workflow

**Original data → Train/Test split → Augment training only → Impute → Scale → GridSearchCV → Test evaluation**

The test data are never augmented.


## 1. Import libraries


In [ ]:

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler

from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor

from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error


## 2. Load the data


In [ ]:
df = pd.read_csv("/mnt/data/gw_wqi(2).csv")

print("Dataset shape:", df.shape)
df.head()


## 3. Prepare the data


In [ ]:

df = df.dropna(axis=1, how="all")

# Remove target/class/identifier columns
columns_to_drop = [
    "wqi_class",
    "iron_(ppm)",
    "year"
]

df = df.drop(
    columns=[c for c in columns_to_drop if c in df.columns]
)

# X = input water-quality variables
# y = groundwater WQI
X = df.select_dtypes(include="number").drop(
    columns=["wqi"],
    errors="ignore"
)

y = df["wqi"]

# Keep only rows with a known WQI
valid = y.notna()

X = X.loc[valid].copy()
y = y.loc[valid].copy()

print("Number of observations:", len(X))
print("Input features:")
print(list(X.columns))


## 4. Split the original data


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Original training observations:", len(X_train))
print("Original testing observations:", len(X_test))


## 5. Augment training data only


In [ ]:

def augment_training_data(X_train, y_train, copies=3):
    X_parts = [X_train.copy()]
    y_parts = [y_train.copy()]

    rng = np.random.RandomState(42)

    for _ in range(copies):
        X_new = X_train.copy()

        for column in X_train.columns:
            std = X_train[column].std()

            if std > 0:
                noise = rng.normal(0, 0.01 * std, len(X_train))
                X_new[column] = X_new[column] + noise

        y_new = y_train + rng.normal(
            0,
            0.005 * y_train.std(),
            len(y_train)
        )

        X_parts.append(X_new)
        y_parts.append(y_new)

    X_aug = pd.concat(X_parts, ignore_index=True)
    y_aug = pd.concat(y_parts, ignore_index=True)

    return X_aug, y_aug


X_train_aug, y_train_aug = augment_training_data(
    X_train,
    y_train
)

print("Training observations after augmentation:", len(X_train_aug))
print("Testing observations:", len(X_test))


## 6. Fill missing values


In [ ]:
imputer = KNNImputer(n_neighbors=5)

X_train_imp = imputer.fit_transform(X_train_aug)
X_test_imp = imputer.transform(X_test)


## 7. Standardize the features


In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_imp)
X_test_scaled = scaler.transform(X_test_imp)


## 8. Define models and hyperparameters


In [ ]:

models = {
    "Decision Tree": DecisionTreeRegressor(random_state=42),

    "Random Forest": RandomForestRegressor(
        random_state=42
    ),

    "Gradient Boosting": GradientBoostingRegressor(
        random_state=42
    ),

    "XGBoost": XGBRegressor(
        random_state=42,
        objective="reg:squarederror"
    ),

    "SVR": SVR()
}

parameter_grids = {
    "Decision Tree": {
        "max_depth": [None, 3, 5, 10],
        "min_samples_split": [2, 5, 10]
    },

    "Random Forest": {
        "n_estimators": [50, 100, 200],
        "max_depth": [None, 5, 10],
        "min_samples_split": [2, 5]
    },

    "Gradient Boosting": {
        "n_estimators": [50, 100, 200],
        "learning_rate": [0.05, 0.1, 0.2],
        "max_depth": [3, 5]
    },

    "XGBoost": {
        "n_estimators": [50, 100, 200],
        "learning_rate": [0.05, 0.1],
        "max_depth": [3, 5, 7]
    },

    "SVR": {
        "C": [1, 10, 100],
        "gamma": ["scale", "auto", 0.1],
        "kernel": ["rbf", "linear"]
    }
}


## 9. Train and evaluate the models


In [ ]:

results = []

for name in models:
    print("Running:", name)

    grid = GridSearchCV(
        estimator=models[name],
        param_grid=parameter_grids[name],
        cv=5,
        scoring="r2",
        n_jobs=-1
    )

    grid.fit(X_train_scaled, y_train_aug)

    y_pred = grid.best_estimator_.predict(X_test_scaled)

    r2 = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)

    nse = 1 - (
        np.sum((y_test - y_pred) ** 2)
        /
        np.sum((y_test - y_test.mean()) ** 2)
    )

    results.append({
        "Model": name,
        "R2": r2,
        "RMSE": rmse,
        "MAE": mae,
        "NSE": nse,
        "Best Parameters": grid.best_params_
    })

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="R2",
    ascending=False
)

results_df


## 10. Check the WQI scale

RMSE and MAE are measured in WQI units.

NRMSE below is only a scale check:

**NRMSE (%) = RMSE / WQI range × 100**


In [ ]:
print("Minimum WQI:", y.min())
print("Maximum WQI:", y.max())
print("Mean WQI:", y.mean())
print("WQI standard deviation:", y.std())

results_df["NRMSE_range_percent"] = (
    results_df["RMSE"]
    / (y.max() - y.min())
    * 100
)

results_df


## 11. Save the final results


In [ ]:
results_df.to_csv(
    "ground_regression_result.csv",
    index=False
)

print("Saved:", "ground_regression_result.csv")
